In [10]:
#Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile

#Extract data
zip_name = 'iris.zip'
zip_ref = zipfile.ZipFile(zip_name, 'r')
zip_ref.extractall()
zip_ref.close()

In [11]:
#Load data
train_df = pd.read_csv('train.csv', index_col='SampleID')
test_df = pd.read_csv('test.csv', index_col='SampleID')

train_df.head()

,sepal_length_(cm),sepal_width_(cm),petal_length_(cm),petal_width_(cm),target
SampleID,,,,,
9,4.4,2.9,1.4,0.2,0
107,4.9,2.5,4.5,1.7,2
77,6.8,2.8,4.8,1.4,1
10,4.9,3.1,1.5,0.1,0
90,5.5,2.5,4.0,1.3,1


In [12]:
#Preprocessing
from sklearn.preprocessing import StandardScaler

X = train_df.drop('target', axis=1)
y = train_df['target']

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

test_scaled = scaler.transform(test_df)
test_scaled = pd.DataFrame(test_scaled, columns=test_df.columns)

In [13]:
#Choose a model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.svm import SVC

def cv_eval(model):
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

models = {
    "LR": LogisticRegression(),
    "RF": RandomForestClassifier(),
    "GB": GradientBoostingClassifier(),
    "HGB": HistGradientBoostingClassifier(),
    "GNB": GaussianNB(),
    "BNB": BernoulliNB(),
    "SVC_linear": SVC(kernel='linear'),
    "SVC_rbf": SVC(kernel='rbf')
}

for name, model in models.items():
    print(f'{name} | f1:{cv_eval(model)}')

LR | f1:0.9579707438530969
RF | f1:0.9496047307812014
GB | f1:0.9579707438530969
HGB | f1:0.941437908496732
GNB | f1:0.9579707438530969
BNB | f1:0.74451564946921
SVC_linear | f1:0.9749019607843138
SVC_rbf | f1:0.9663367569249923


In [14]:
#Fine-tune
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

model_pipe = Pipeline(steps=[
    ('m', SVC(kernel='linear'))
])

param_grid = {
    'm__C': [0.1, 1, 10, 100],
    'm__gamma': [1, 0.1, 0.01, 0.001]
}

search = GridSearchCV(model_pipe, param_grid, cv=5, verbose=1, n_jobs=-1, scoring='f1_macro')
search.fit(X_scaled, y)

best_model = search.best_estimator_
print('Best score:',  search.best_score_)
print('Best params:', search.best_params_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best score: 0.9749019607843138
Best params: {'m__C': 1, 'm__gamma': 1}


In [15]:
#Submit
best_model.fit(X_scaled, y)
preds = best_model.predict(test_scaled)

output_df = pd.DataFrame({
    'SampleID': test_df.index,
    'label': preds
})

output_df.head()

,SampleID,label
0,39,0
1,128,2
2,58,1
3,94,1
4,43,0


In [16]:
output_df.to_csv('submission.csv', index=False)

The idea was to use linear models(SVC)